# GlyphLM Experiment

Does shorthand compression grounded in real stenography make LLM tokenizers and training more efficient? This notebook runs the full pipeline: encode -> tokenize -> train -> eval, comparing a tiny GPT-2 trained on raw `tiny_shakespeare` text against one trained on a shorthand-compressed version, where the shorthand comes from real chords in [Plover](https://github.com/openstenoproject/plover), an open-source stenography engine.

Related work: [Training LLMs over Neurally Compressed Text](https://arxiv.org/abs/2404.03626) (Lester et al., 2024). Full build/iteration log: `docs/log.md`.

Runs end-to-end on Google Colab free tier (T4 GPU).

In [ ]:
!pip install -q torch transformers tokenizers datasets numpy

Clone the repo to get the `glyph` package. **Note:** if `shoegazerstella/glyph-lm` is private, an anonymous Colab runtime cannot clone it without a GitHub personal access token (`https://<token>@github.com/shoegazerstella/glyph-lm.git`), or you can make the repo public before running this notebook on Colab.

In [ ]:
!git clone https://github.com/shoegazerstella/glyph-lm.git
%cd glyph-lm

## Step 1: Shorthand Encoding

Fits a word/phrase -> real-steno-chord mapping from the training corpus itself (most frequent words/phrases that have an actual Plover chord), then applies it to produce the glyph corpus. Measures the resulting whitespace-token compression ratio.

In [ ]:
from glyph import data

data.build_corpora()

## Step 2: Tokenizer Training

Trains two BPE tokenizers (vocab_size=2048) — one on raw text, one on glyph text — and reports vocab size, average tokens/line, and compression ratio vs. character count for each.

In [ ]:
from glyph import tokenizer as tok_mod

raw_tok = tok_mod.train_bpe("data/raw/train.txt", "tokenizers/raw_bpe")
glyph_tok = tok_mod.train_bpe("data/glyph/train.txt", "tokenizers/glyph_bpe")
tok_mod.report_stats(raw_tok, "data/raw/train.txt", "raw")
tok_mod.report_stats(glyph_tok, "data/glyph/train.txt", "glyph")

## Step 3: Model Training

Trains two identical tiny GPT-2 models (n_embd=128, n_layer=4, n_head=4) for 3 epochs each — one on raw text, one on glyph text — using MPS on Apple Silicon, CUDA on Colab's T4, or CPU as a fallback.

In [ ]:
from glyph import train

train.train_model("data/raw/train.txt", "tokenizers/raw_bpe", "models/model_raw", "raw")
train.train_model("data/glyph/train.txt", "tokenizers/glyph_bpe", "models/model_glyph", "glyph")

## Step 4: Evaluation

Compares both models on held-out perplexity, inference tokens/second, average tokens per line, and compression ratio — plus 3 sample completions each from the same seed prompt.

In [ ]:
from glyph import eval as eval_mod

eval_mod.main()

## Results & Interpretation

**Hypothesis recap:** at equal token budget, glyph-compressed text should yield a better compression ratio and comparable-or-better perplexity/tokens-per-second than raw text.

**Observed results (first run, Apple M5, ~1 min total compute):**

| Metric | raw | glyph |
|---|---|---|
| Final train loss | 6.008 | 5.842 |
| Perplexity (val set) | 408.8 | 393.6 |
| Tokens/second (inference) | 169.9 | 289.9 |
| Compression ratio (tokens/char) | 0.333 | 0.358 |
| Whitespace-token ratio (glyph/raw) | 1.0 | |

**Discussion:** glyph text trained to lower loss, better perplexity, and much faster inference, but its character-level BPE compression ratio was actually worse, and the whitespace-token ratio was exactly 1.0. Both are explainable: the encoder swaps one word for one glyph, so it never merges words into fewer tokens; and the novel glyph symbols are rare, so they compete poorly for a fixed 2048-slot vocab against common English morphemes.

**Limitations:** this was a ~114-step smoke test on a 1MB corpus to prove the pipeline runs end-to-end, not a converged, statistically meaningful comparison. Next steps: encode common phrases (not just single words) to actually shrink token counts, try SentencePiece Unigram instead of BPE, and train longer on more data before drawing real conclusions.